# 예제 07. Excel 업무 자동화 에이전트
### 반복적인 고객 데이터 집계와 Excel 보고서 작성 자동화 · 2일 차 · 60분

> **실행 방법**: 위에서부터 순서대로 실행하세요. 셀을 클릭하고 `Shift + Enter`. AI 답변이 나오는 셀은 몇 초~수십 초 걸립니다. 셀 왼쪽이 `[*]`이면 아직 실행 중입니다.

| 항목 | 내용 |
|---|---|
| 데이터 | `data/bank_customers.csv` (UCI Bank Marketing 45,211건) |
| 오늘의 결과물 | 고객군별 정기예금 가입 현황 Excel 보고서 `outputs/고객별_예금가입_분석.xlsx` |

**오늘 해결할 질문**: "매달 손으로 만들던 고객군별 집계표를, 파일만 바꿔 넣으면 자동으로 만들 수 있을까?"

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

In [ ]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

## 07-1. 고객 데이터 불러오기와 기본 정보

`load_customers()`는 CSV를 읽고 1일 차와 **같은 기준**으로 연령대 · 직업 · 잔액구간 · 대출상태 항목을 붙여 줍니다.

In [ ]:
from exercises.common import load_customers, overview, rate_table

df = load_customers(DATA)
print(overview(df))
df[["age", "연령대", "직업", "balance", "잔액구간", "대출상태", "y"]].head()

## 07-2. 연령대별 · 직업군별 가입률 집계

Excel에 들어갈 표를 Python이 먼저 계산합니다. 1일 차 예제 02에서 본 숫자와 같습니다.

In [ ]:
rate_table(df, "연령대")

In [ ]:
rate_table(df, "직업군")

## 07-3. 잔액 구간별 · 대출 보유별 가입률

In [ ]:
rate_table(df, "잔액구간")

In [ ]:
rate_table(df, "대출상태")

## 07-4. 에이전트로 Excel 보고서 만들기 ⭐

질문 한 줄로 **집계 → AI 해석 → Excel 저장**이 한 번에 실행됩니다.

In [ ]:
결과 = agent.run("ex07", DATA, "고객군별 정기예금 가입률을 분석하고 특이사항과 실무 아이디어를 정리해 주세요.")

> 💬 AI 답변의 문장은 PC와 모델에 따라 조금씩 다릅니다. 표·숫자·파일은 Python이 계산하므로 모두 같게 나옵니다.

## 07-5. 만들어진 Excel 파일 확인

파일 탐색기에서 실습 폴더의 `outputs/고객별_예금가입_분석.xlsx`를 더블클릭해 엽니다. 아래 셀은 시트 구성만 확인합니다.

In [ ]:
import pandas as pd

엑셀 = pd.ExcelFile("outputs/고객별_예금가입_분석.xlsx")
print("시트:", 엑셀.sheet_names)
pd.read_excel(엑셀, sheet_name="직업군", header=3).head()

**Excel 보고서 구성** — 사람이 매달 손으로 하던 서식 작업을 코드가 대신합니다.

| 시트 | 내용 | 자동으로 적용된 것 |
|---|---|---|
| 요약 | 작성일, 데이터 출처, 전체 고객 수·가입자 수·가입률, AI 분석 의견, 숫자 검증 결과 | 제목 서식, 숫자 형식(#,##0 / 0.0) |
| 연령대 · 직업군 · 잔액구간 · 대출상태 | 고객수 · 가입자수 · 가입률 표 | **표 스타일**, **데이터 막대**(가입률 길이), **조건부 서식**(전체 평균보다 높으면 초록), **막대 차트** |

## 07-6. [선택] 🔧 시트 구성 바꾸기

`DIMENSIONS` 목록을 바꾸면 Excel 시트가 바뀝니다. 쓸 수 있는 기준: `"연령대"`, `"직업군"`, `"잔액구간"`, `"대출상태"`, `"과거캠페인"`

In [ ]:
from exercises import ex07_excel

ex07_excel.DIMENSIONS = ["연령대", "잔액구간", "과거캠페인"]     # ← 바꿔 보세요
결과 = agent.run("ex07", DATA, "고객군별 정기예금 가입률을 분석해 주세요.")
ex07_excel.DIMENSIONS = ["연령대", "직업군", "잔액구간", "대출상태"]   # 원래대로

## 07-7. 입력 파일을 바꿔 같은 작업 반복하기

자동화의 진짜 힘은 **다음 달 파일만 바꿔 넣으면 같은 보고서가 나온다**는 데 있습니다.
여기서는 "휴대폰으로 연락한 고객"만 뽑은 새 파일을 만들어, 같은 에이전트에 넣어 봅니다.

In [ ]:
원본 = pd.read_csv(DATA)
원본[원본["contact"] == "cellular"].to_csv("data/bank_customers_cellular.csv", index=False)

결과 = agent.run("ex07", "data/bank_customers_cellular.csv", "휴대폰 연락 고객의 고객군별 정기예금 가입률을 분석해 주세요.")

> 💡 **핵심 정리** — 파일 이름이 바뀌면 결과 파일도 `고객별_예금가입_분석_bank_customers_cellular.xlsx`로 따로 저장됩니다.
> 휴대폰 연락 고객은 29,285명, 가입률 14.9%로 전체(11.7%)보다 높습니다. 코드는 한 글자도 바꾸지 않았습니다.

## 07-8. AI 해석 검토하기 ⚠️

결과 아래 **숫자 검증** 줄을 확인합니다.
- ✅ → AI 문장 속 숫자가 모두 Pandas 계산 결과에 있음
- ⚠️ → 표에 없는 숫자가 있음. AI가 직접 계산(예: "약 3배")했거나 지어낸 숫자일 수 있으니 원본 표와 대조

Excel의 **요약 시트**에도 같은 검증 결과가 함께 저장됩니다. 보고서를 위에 올리기 전, 사람이 마지막으로 확인합니다.

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 고객 데이터 + 분석 항목 | `load_customers(파일)` |
| 고객군별 가입률 표 | `rate_table(df, "연령대")` |
| 집계 → AI 해석 → Excel | `agent.run("ex07", 파일, 질문)` |
| 시트 구성 바꾸기 | `ex07_excel.DIMENSIONS = [...]` |
| 일부 고객만 새 파일로 | `원본[조건].to_csv("data/새파일.csv", index=False)` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션 1.** 미혼(`marital == "single"`) 고객만 뽑은 파일을 만들어 Excel 보고서를 만들어 보세요.
**이렇게 나오면 성공!** `outputs/고객별_예금가입_분석_bank_customers_single.xlsx` 생성 · 요약 시트의 전체 고객수 12,790 · 가입률 14.9

**미션 2.** 만든 Excel을 열어 **초록색**으로 표시된(전체 평균보다 가입률이 높은) 연령대를 찾아보세요.
**이렇게 나오면 성공!** 20대 이하 21.0% · 60대 이상 21.3% (미혼 평균 14.9%보다 높음). 단, 60대 이상 미혼 고객은 61명뿐이라 해석에 주의

**다음 예제 전에 이것만은 꼭!**
- [ ] 같은 코드 + 다른 파일 = 같은 형식의 보고서
- [ ] Excel에 저장된 AI 의견은 숫자 검증 결과와 함께 검토

In [ ]:
# 미션 1 — 07-7 셀과 같은 모양입니다
원본[원본["contact"] == "cellular"].to_csv("data/bank_customers_cellular.csv", index=False)   # ← "marital" · "single"로
결과 = agent.run("ex07", "data/bank_customers_cellular.csv",       # ← 파일 이름도 single로
                 "휴대폰 연락 고객의 고객군별 정기예금 가입률을 분석해 주세요.")   # ← "미혼 고객의"로

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
# 미션 1 정답
원본[원본["marital"] == "single"].to_csv("data/bank_customers_single.csv", index=False)
결과 = agent.run("ex07", "data/bank_customers_single.csv", "미혼 고객의 고객군별 정기예금 가입률을 분석해 주세요.")

---
### ▶ 다음 예제 09. 금융 업무보고서 · 공문 작성 에이전트
지금까지의 표와 AI 해석을 모아 **Word 업무보고서와 내부 기안문**을 만듭니다.

(예제 08 차트 자동 생성은 시간이 남을 때 하는 선택 예제입니다.)